# Entity resolution with derived confidence

Drug mentions to RxNorm, disease mentions to MONDO. Replaces `scripts/resolve_diseases.py`.

No confidence value in this system is supplied by a caller — it is a property of the method that produced it, and a lint rule fails the build on any literal.

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## The problem

ClinicalTrials.gov condition strings carry no identifier, so one node per surface form. Six variants of type 2 diabetes, fragmenting the trial count across all of them.

In [2]:
from biomedical.ui import data

print("canonical diseases, by trial count:")
for r in data.q("""MATCH (t:ClinicalTrial)-[:STUDIES]->(d:Disease)
                   WHERE d.mondo_id IS NOT NULL
                   RETURN d.name AS n, d.mondo_id AS id, count(DISTINCT t) AS t
                   ORDER BY t DESC LIMIT 8"""):
    print(f"  {r['n'][:40]:<40} {r['id']:<16} {r['t']:>4} trials")

canonical diseases, by trial count:


  rheumatoid arthritis                     MONDO:0008383     165 trials
  type 2 diabetes mellitus                 MONDO:0005148     159 trials
  atrial fibrillation                      MONDO:0004981     153 trials
  breast cancer                            MONDO:0007254     126 trials
  psoriasis                                MONDO:0005083      89 trials
  non-small cell lung carcinoma            MONDO:0005233      71 trials
  lung cancer                              MONDO:0008903      49 trials
  psoriatic arthritis                      MONDO:0011849      40 trials


## Surface forms collapsed onto one identifier

In [3]:
for r in data.q("""MATCH (m:Mention {mention_type:'condition'})-[:RESOLVES_TO]->(d:Disease)
                   WITH d, count(m) AS v WHERE v > 2
                   RETURN d.name AS n, d.mondo_id AS id, v ORDER BY v DESC LIMIT 8"""):
    print(f"  {r['n'][:40]:<40} {r['id']:<16} {r['v']:>3} surface forms")

print("\nthe forms behind MONDO:0005148:")
for r in data.q("""MATCH (m:Mention)-[r:RESOLVES_TO]->(:Disease {mondo_id:'MONDO:0005148'})
                   RETURN m.raw_text AS raw, r.method AS m, r.confidence AS c
                   ORDER BY r.confidence DESC LIMIT 14"""):
    print(f"  {r['raw'][:44]:<44} {r['c']}  {r['m']}")

  non-small cell lung carcinoma            MONDO:0005233     19 surface forms
  type 2 diabetes mellitus                 MONDO:0005148     13 surface forms
  breast cancer                            MONDO:0007254      7 surface forms
  melanoma                                 MONDO:0005105      5 surface forms
  non-small cell squamous lung carcinoma   MONDO:0056806      5 surface forms
  lung cancer                              MONDO:0008903      5 surface forms
  coronary artery disorder                 MONDO:0005010      5 surface forms
  rheumatoid arthritis                     MONDO:0008383      5 surface forms

the forms behind MONDO:0005148:


  Type 2 Diabetes Mellitus                     1.0  exact_identifier
  Type 2 Diabetes Mellitus (T2DM)              0.95  normalised_exact
  Type2 Diabetes Mellitus                      0.95  normalised_exact
  Type 2 Diabetes                              0.9  exact_synonym_match
  Diabetes Mellitus, Type 2                    0.9  exact_synonym_match
  Type II Diabetes                             0.9  exact_synonym_match
  Diabetes Mellitus, Type II                   0.9  exact_synonym_match
  Type2 Diabetes                               0.9  exact_synonym_match
  Type II Diabetes Mellitus                    0.9  exact_synonym_match
  Type 2 Diabetes (T2DM)                       0.9  exact_synonym_match
  Diabetes Type 2 on Insulin                   0.6  single_fuzzy_above_threshold
  Diabetes Mellitus, Non-Insulin-Dependent     0.6  single_fuzzy_above_threshold
  Non-Insulin-Dependent Diabetes Mellitus      0.6  single_fuzzy_above_threshold


## Normalisation rules, each individually tested

Silent resolution errors originate here far more often than in matching. The stage/severity strip is **lossy** — a deliberate trade-off, since it merges clinically distinct cohorts onto one concept.

In [4]:
from biomedical.sources.resolver import normalise_disease

for s in ["Diabetes Mellitus, Type 2", "Diabetes Mellitus Type 2",
          "Type2 Diabetes Mellitus", "Type 2 Diabetes Mellitus (T2DM)",
          "IBD - Inflammatory Bowel Disease", "COPD: Chronic Obstructive Pulmonary Disease",
          "Carcinoma, Non-Small-Cell Lung", "Metastatic Melanoma"]:
    print(f"  {s:<46} -> {normalise_disease(s)!r}")

  Diabetes Mellitus, Type 2                      -> 'type 2 diabetes mellitus'
  Diabetes Mellitus Type 2                       -> 'type 2 diabetes mellitus'
  Type2 Diabetes Mellitus                        -> 'type 2 diabetes mellitus'
  Type 2 Diabetes Mellitus (T2DM)                -> 'type 2 diabetes mellitus'
  IBD - Inflammatory Bowel Disease               -> 'inflammatory bowel disease'
  COPD: Chronic Obstructive Pulmonary Disease    -> 'chronic obstructive pulmonary disease'
  Carcinoma, Non-Small-Cell Lung                 -> 'non-small-cell lung carcinoma'
  Metastatic Melanoma                            -> 'melanoma'


## The cascade, live

Stages 4 and 5 sit below the trust-boundary floor of 0.75, so they can never become rule inputs without human review.

In [5]:
from biomedical.sources.resolver import resolve_disease, resolve_drug

print("DISEASES")
for q in ["Rheumatoid Arthritis", "Diabetes Mellitus, Type 2",
          "IBD - Inflammatory Bowel Disease", "Advanced Solid Tumor"]:
    print("  " + resolve_disease(q).describe())

print("\nDRUGS (human-entered FAERS strings)")
for q in ["ADALIMUMAB", "ASPIRIN.", "ABRAXANE (PACLITAXEL) (UNKNOWN)"]:
    print("  " + resolve_drug(q).describe())

DISEASES


  Rheumatoid Arthritis                     -> rheumatoid arthritis (MONDO:0008383)       1.00 via exact_identifier


  Diabetes Mellitus, Type 2                -> type 2 diabetes mellitus (MONDO:0005148)   0.90 via exact_synonym_match


  IBD - Inflammatory Bowel Disease         -> inflammatory bowel disease (MONDO:0005265) 0.95 via normalised_exact


  Advanced Solid Tumor                     -> RET fusion positive cancer (MONDO:0700216) 0.30 via multiple_candidates  [NEEDS_REVIEW]

DRUGS (human-entered FAERS strings)


  ADALIMUMAB                                 -> adalimumab (327361)                      1.00 via exact_identifier


  ASPIRIN.                                   -> aspirin (1191)                           0.95 via normalised_exact


  ABRAXANE (PACLITAXEL) (UNKNOWN)            -> paclitaxel Injection [Abraxane] (1663184 0.30 via multiple_candidates  [NEEDS_REVIEW]


## Confidence distribution and the review backlog

The backlog is published deliberately. A visible one is more credible than none.

In [6]:
for r in data.resolution_bands():
    print(f"  {str(r['method']):<36} n={r['count']:<5} conf={r['confidence']}")

b = data.backlog_counts()
print(f"\n  resolved {b['resolved']}   needs_review {b['needs_review']}"
      f"   unlinked {b['unlinked']}   cleared {b['cleared']}   superseded {b['superseded']}")

  exact_identifier                     n=196   conf=1.0
  multiple_candidates                  n=140   conf=0.3
  exact_synonym_match                  n=106   conf=0.9
  normalised_exact                     n=56    conf=0.95
  single_fuzzy_above_threshold         n=55    conf=0.6
  unique_ingredient_after_dose_strip   n=1     conf=0.9
  human_review                         n=1     conf=1.0



  resolved 555   needs_review 178   unlinked 18   cleared 1   superseded 1


## Three defects a manual audit found in this code

**1. Fabricated confidence.** OLS4 returns no relevance score — `None` on every document. The first version defaulted it to `1.0`, so every candidate scored perfectly and the fuzzy band was meaningless. Exactly the defect this project exists to prevent, committed in the module whose docstring promises derived confidence.

Two serious false positives were corrected once scores became real:

| Raw | Was | Now |
|---|---|---|
| `Sjogren's Syndrome` | **Marinesco-Sjogren syndrome** | Sjogren syndrome |
| `Stroke` | **MELAS syndrome** | stroke disorder |

**2. Veterinary terms.** `Bradycardia` resolved to `sinus bradycardia, non-human animal`.

**3. Token-subset bonus too generous.** `solid tumor` scored 0.85 against `NTRK fusion positive cancer`.

In [7]:
from biomedical.sources.mondo import MondoTerm, best_similarity, is_non_human, search

print("similarity is computed locally now, because OLS4 returns no score:")
for q in ["chronic kidney disease", "solid tumor", "inflammatory bowel disease"]:
    print(f"  {q!r}")
    for t, sc in search(q, rows=8)[:3]:
        print(f"      {sc:.3f}  {t.label[:52]}")

print("\nveterinary filter:")
for lbl in ["sinus bradycardia, non-human animal", "type 2 diabetes mellitus, pig",
            "catscratch disease", "pigmented villonodular synovitis"]:
    print(f"  rejected={is_non_human(MondoTerm('X', lbl))!s:<6} {lbl}")

similarity is computed locally now, because OLS4 returns no score:
  'chronic kidney disease'


      1.000  chronic kidney disease
      0.830  end stage renal failure
      0.791  chronic renal failure syndrome
  'solid tumor'


      0.524  RET fusion positive cancer
      0.522  solid pseudopapillary neoplasm of the pancreas
      0.512  NTRK fusion positive cancer
  'inflammatory bowel disease'


      1.000  inflammatory bowel disease
      0.963  inflammatory bowel disease 1
      0.946  inflammatory bowel disease 28

veterinary filter:
  rejected=True   sinus bradycardia, non-human animal
  rejected=True   type 2 diabetes mellitus, pig
  rejected=False  catscratch disease
  rejected=False  pigmented villonodular synovitis


## Evaluating a matcher with no gold standard

No answer key exists for these strings. Round-trip detects **definite errors**; agreement establishes nothing about accuracy, and the report says so.

Drug round-trip agreement by slice ranged **57%–93%** — resolution quality is clearly domain-dependent.

A limitation no threshold fixes: **MONDO is a disease ontology; many trial conditions are phenotypes.** `Bradycardia` and `Albuminuria` belong in HPO. That needs a second authority, which is not built.

In [8]:
from biomedical.sources.resolver import evaluate_consistency, resolve_drug

sample = [resolve_drug(x) for x in ["ADALIMUMAB", "ASPIRIN", "ATROPINE", "AVONEX"]]
print(evaluate_consistency(sample).describe())

resolution quality, evaluated WITHOUT a gold standard:
  round-trip agreement: 4/4 = 100%
  confidence bands:
    exact_identifier                   4
  establishes: definite errors (a disagreeing round-trip is wrong).
  does NOT establish: accuracy. No answer key exists for these strings.
